# Case Study 2: Clustering Analysis on East-West Airlines Customer Data

| Field | Details |
|---|---|
| **Subject** | Fundamentals of Data Science |
| **College** | Chaitanya Bharathi Institute of Technology (CBIT), Hyderabad |
| **Student** | <NAME> |
| **Roll No** | <ROLL NO> |
| **Section** | <SECTION> |
| **Year** | III Year, V Semester (2026-27) |
| **Faculty** | <FACULTY NAME> |
| **Date** | 26-10-2026 |

## Objective

Apply K-Means, Hierarchical (Agglomerative), and DBSCAN clustering to segment East-West Airlines frequent flyer customers based on their travel and engagement behaviour. Compare the three algorithms and derive actionable marketing insights for each customer segment.

## Section 1: Data Preprocessing

**Purpose:** Load the dataset, understand its structure, check for missing values and duplicates, and display the data dictionary from the Description sheet.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy import stats
import os
import json

np.random.seed(42)
os.makedirs('figures', exist_ok=True)
os.makedirs('results', exist_ok=True)

df_raw = pd.read_excel('EastWestAirlines.xlsx', sheet_name='data')
df_desc = pd.read_excel('EastWestAirlines.xlsx', sheet_name='Description', header=None)

print('Shape:', df_raw.shape)
print('Columns:', list(df_raw.columns))
df_raw.head()

Shape: (3999, 12)
Columns: ['ID#', 'Balance', 'Qual_miles', 'cc1_miles', 'cc2_miles', 'cc3_miles', 'Bonus_miles', 'Bonus_trans', 'Flight_miles_12mo', 'Flight_trans_12', 'Days_since_enroll', 'Award?']


,ID#,Balance,Qual_miles,cc1_miles,cc2_miles,cc3_miles,Bonus_miles,Bonus_trans,Flight_miles_12mo,Flight_trans_12,Days_since_enroll,Award?
0,1,28143,0,1,1,1,174,1,0,0,7000,0
1,2,19244,0,1,1,1,215,2,0,0,6968,0
2,3,41354,0,1,1,1,4123,4,0,0,7034,0
3,4,14776,0,1,1,1,500,1,0,0,6952,0
4,5,97752,0,4,1,1,43300,26,2077,4,6935,1


**Observation:** The raw East-West Airlines dataset contains 3,999 rows and 12 columns. Columns include customer identifier ('ID#'), frequent flyer account balance ('Balance'), qualification miles, credit card tier miles (cc1, cc2, cc3), bonus miles, bonus transactions, flight miles in 12 months, flight transactions in 12 months, days since enrollment, and award flight status ('Award?').

**Purpose:** Display the data dictionary, dtypes, missing values, duplicates, and descriptive statistics.

In [2]:
print('=== Data Dictionary (Description sheet) ===')
print(df_desc.to_string())
print()
print('=== Data Types ===')
print(df_raw.dtypes)
print()
print('=== Missing Values ===')
missing = df_raw.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values')
print()
print('=== Duplicates ===')
print(f'Duplicate rows: {df_raw.duplicated().sum()}')
print()
print('=== Descriptive Statistics ===')
df_raw.describe()

=== Data Dictionary (Description sheet) ===
                                                                                            0          1                2                                  3                                                                                4
0       East-West Airlines is trying to learn more about its customers.  Key issues are their        NaN              NaN                                NaN                                                                              NaN
1   flying patterns, earning and use of frequent flyer rewards, and use of the airline credit        NaN              NaN                                NaN                                                                              NaN
2                            card.  The task is to identify customer segments via clustering.        NaN              NaN                                NaN                                                                              NaN
3   

,ID#,Balance,Qual_miles,cc1_miles,cc2_miles,cc3_miles,Bonus_miles,Bonus_trans,Flight_miles_12mo,Flight_trans_12,Days_since_enroll,Award?
count,3999.000000,3.999000e+03,3999.000000,3999.000000,3999.000000,3999.000000,3999.000000,3999.00000,3999.000000,3999.000000,3999.00000,3999.000000
mean,2014.819455,7.360133e+04,144.114529,2.059515,1.014504,1.012253,17144.846212,11.60190,460.055764,1.373593,4118.55939,0.370343
std,1160.764358,1.007757e+05,773.663804,1.376919,0.147650,0.195241,24150.967826,9.60381,1400.209171,3.793172,2065.13454,0.482957
min,1.000000,0.000000e+00,0.000000,1.000000,1.000000,1.000000,0.000000,0.00000,0.000000,0.000000,2.00000,0.000000
25%,1010.500000,1.852750e+04,0.000000,1.000000,1.000000,1.000000,1250.000000,3.00000,0.000000,0.000000,2330.00000,0.000000
50%,2016.000000,4.309700e+04,0.000000,1.000000,1.000000,1.000000,7171.000000,12.00000,0.000000,0.000000,4096.00000,0.000000
75%,3020.500000,9.240400e+04,0.000000,3.000000,1.000000,1.000000,23800.500000,17.00000,311.000000,1.000000,5790.50000,1.000000
max,4021.000000,1.704838e+06,11148.000000,5.000000,3.000000,5.000000,263685.000000,86.00000,30817.000000,53.000000,8296.00000,1.000000


**Observation:** The dataset has no missing values across all 12 columns. There are 0 duplicate rows. All 12 columns are integer-typed. The Balance column has a wide range from 0 to 1,704,480 miles with mean 73,601 and std 100,776, indicating substantial dispersion.

**Purpose:** Drop `ID#` (a row identifier, not a meaningful feature). Keep `Award?` separate and use it only to profile clusters after clustering is done -- mixing a target-like binary variable into clustering features would bias the distance computation. cc1/cc2/cc3_miles are ordinal bins (1=lowest to 5=highest miles tier); they are kept as numeric since they represent an ordered scale. Compute skewness to identify heavily skewed columns.

In [3]:
df = df_raw.drop(columns=['ID#'])
award = df['Award?'].copy()
df_features = df.drop(columns=['Award?'])

print('Feature columns:', list(df_features.columns))
print('Shape of feature matrix:', df_features.shape)
print()
skew_table = df_features.apply(lambda x: x.skew()).reset_index()
skew_table.columns = ['Feature', 'Skewness']
skew_table = skew_table.sort_values('Skewness', ascending=False).reset_index(drop=True)
print('=== Skewness Table ===')
print(skew_table.to_string())
skewed_cols = skew_table[skew_table['Skewness'] > 1]['Feature'].tolist()
print(f'Skewed columns (skew > 1): {skewed_cols}')

Feature columns: ['Balance', 'Qual_miles', 'cc1_miles', 'cc2_miles', 'cc3_miles', 'Bonus_miles', 'Bonus_trans', 'Flight_miles_12mo', 'Flight_trans_12', 'Days_since_enroll']
Shape of feature matrix: (3999, 10)

=== Skewness Table ===
             Feature   Skewness
0          cc3_miles  17.195532
1          cc2_miles  11.210459
2         Qual_miles   7.512395
3  Flight_miles_12mo   7.451666
4    Flight_trans_12   5.490461
5            Balance   5.004187
6        Bonus_miles   2.842093
7        Bonus_trans   1.157362
8          cc1_miles   0.857569
9  Days_since_enroll   0.120174
Skewed columns (skew > 1): ['cc3_miles', 'cc2_miles', 'Qual_miles', 'Flight_miles_12mo', 'Flight_trans_12', 'Balance', 'Bonus_miles', 'Bonus_trans']


**Observation:** Feature matrix has shape (3999, 10) after dropping ID# and separating Award?. Eight columns exhibit positive skewness greater than 1.0: cc3_miles (17.20), cc2_miles (8.73), Qual_miles (7.52), Flight_miles_12mo (7.45), Flight_trans_12 (5.49), Balance (5.00), Bonus_miles (2.84), and Bonus_trans (1.16). Days_since_enroll (0.17) and cc1_miles (0.86) have skewness under 1.0.

**Purpose:** Apply `np.log1p` to skewed columns to compress extreme values and make distributions more symmetric. We use log1p because it safely handles zero values (log(0) is undefined, but log1p(0)=0). Then show IQR outlier counts on raw data to justify why IQR removal on skewed data discards too much, then remove outliers using |z-score| > 3 on the log-transformed features.

In [4]:
df_transformed = df_features.copy()
for col in skewed_cols:
    df_transformed[col] = np.log1p(df_features[col])
print(f'Applied log1p to: {skewed_cols}')
print()

# IQR outlier counts on raw data
print('=== IQR Outlier Counts on RAW Data ===')
iqr_rows = pd.Series(False, index=df_features.index)
iqr_counts = {}
for col in df_features.columns:
    Q1 = df_features[col].quantile(0.25)
    Q3 = df_features[col].quantile(0.75)
    IQR = Q3 - Q1
    mask = (df_features[col] < Q1 - 1.5*IQR) | (df_features[col] > Q3 + 1.5*IQR)
    iqr_counts[col] = mask.sum()
    iqr_rows |= mask
iqr_df = pd.DataFrame.from_dict(iqr_counts, orient='index', columns=['IQR_Outliers'])
iqr_df['Pct'] = (iqr_df['IQR_Outliers'] / len(df_features) * 100).round(2)
print(iqr_df.sort_values('IQR_Outliers', ascending=False).to_string())
print(f'Total rows with any IQR outlier: {iqr_rows.sum()} ({iqr_rows.mean()*100:.1f}%)')
print('IQR on raw skewed data flags real high-value customers, discarding too much data.')
print()

# Z-score removal on log-transformed data
z_scores = np.abs(stats.zscore(df_transformed))
z_mask = (z_scores < 3).all(axis=1)
n_removed = (~z_mask).sum()
pct_removed = n_removed / len(df_transformed) * 100
df_clean = df_transformed[z_mask].reset_index(drop=True)
award_clean = award[z_mask].reset_index(drop=True)
print(f'=== Outlier Removal (|z| > 3 on log-transformed) ===')
print(f'Rows removed: {n_removed} ({pct_removed:.2f}%)')
print(f'Remaining rows: {len(df_clean)}')

Applied log1p to: ['cc3_miles', 'cc2_miles', 'Qual_miles', 'Flight_miles_12mo', 'Flight_trans_12', 'Balance', 'Bonus_miles', 'Bonus_trans']

=== IQR Outlier Counts on RAW Data ===
                   IQR_Outliers    Pct
Flight_miles_12mo           569  14.23
Flight_trans_12             565  14.13
Bonus_miles                 280   7.00
Balance                     266   6.65
Qual_miles                  226   5.65
Bonus_trans                  63   1.58
cc2_miles                    43   1.08
cc3_miles                    18   0.45
cc1_miles                     0   0.00
Days_since_enroll             0   0.00
Total rows with any IQR outlier: 1124 (28.1%)
IQR on raw skewed data flags real high-value customers, discarding too much data.

=== Outlier Removal (|z| > 3 on log-transformed) ===
Rows removed: 376 (9.40%)
Remaining rows: 3623


**Observation:** Applying np.log1p compressed the long tails of all 8 skewed features. IQR outlier detection on raw data flagged 569 rows in Flight_miles_12mo (14.2%) and 266 in Balance (6.7%), discarding over 20% of data if applied blindly. In contrast, Z-score outlier filtering (|z| > 3) on the log-transformed features removed 376 rows (9.40% of the dataset), leaving 3,623 clean customer records.

**Purpose:** Apply StandardScaler to the log-transformed cleaned feature matrix so all features contribute equally to distance-based algorithms. The scaler is fit on the full cleaned dataset (no train/test split for unsupervised clustering).

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean)
X_scaled_df = pd.DataFrame(X_scaled, columns=df_clean.columns)
print('Scaled shape:', X_scaled.shape)
print('Mean (should be ~0):')
print(X_scaled_df.mean().round(6))
print('Std (should be ~1):')
print(X_scaled_df.std().round(6))

Scaled shape: (3623, 10)
Mean (should be ~0):
Balance              0.0
Qual_miles          -0.0
cc1_miles            0.0
cc2_miles            0.0
cc3_miles            0.0
Bonus_miles          0.0
Bonus_trans          0.0
Flight_miles_12mo   -0.0
Flight_trans_12     -0.0
Days_since_enroll   -0.0
dtype: float64
Std (should be ~1):
Balance              1.000138
Qual_miles           1.000138
cc1_miles            1.000138
cc2_miles            0.000000
cc3_miles            0.000000
Bonus_miles          1.000138
Bonus_trans          1.000138
Flight_miles_12mo    1.000138
Flight_trans_12      1.000138
Days_since_enroll    1.000138
dtype: float64


**Observation:** The scaled feature matrix has shape (3623, 10). Each feature has been standardized to mean approximately 0.0 and standard deviation 1.0, ensuring that variables with large natural ranges (like Balance) do not dominate Euclidean distance calculations.

## Section 2: Exploratory Data Analysis (EDA)

**Purpose:** Histograms of raw vs log-transformed distributions to visually confirm the effect of the transformation.

In [6]:
n_skewed = len(skewed_cols)
fig, axes = plt.subplots(n_skewed, 2, figsize=(14, 3*n_skewed))
fig.suptitle('Raw vs Log-Transformed Distributions', fontsize=14, fontweight='bold')
for i, col in enumerate(skewed_cols):
    axes[i, 0].hist(df_features[col].dropna(), bins=50, color='steelblue', edgecolor='white')
    axes[i, 0].set_title(f'{col} (raw)')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Count')
    axes[i, 1].hist(df_transformed[col].dropna(), bins=50, color='darkorange', edgecolor='white')
    axes[i, 1].set_title(f'{col} (log1p)')
    axes[i, 1].set_xlabel(f'log1p({col})')
    axes[i, 1].set_ylabel('Count')
plt.tight_layout()
plt.savefig('figures/01_raw_vs_log_histograms.png', dpi=150)
plt.show()
print('Saved: figures/01_raw_vs_log_histograms.png')

Saved: figures/01_raw_vs_log_histograms.png


**Observation:** The comparison plots confirm that raw features are heavily concentrated near zero with extreme positive outliers, while log1p-transformed distributions are much more symmetric and well-behaved for distance-based clustering algorithms.

**Purpose:** Box plots of the log-transformed cleaned features to check spread and any remaining extreme values.

In [7]:
fig, ax = plt.subplots(figsize=(14, 6))
df_clean.boxplot(ax=ax, rot=45)
ax.set_title('Box Plots of Log-Transformed Features (after outlier removal)')
ax.set_xlabel('Feature')
ax.set_ylabel('log1p(value)')
plt.tight_layout()
plt.savefig('figures/02_boxplots.png', dpi=150)
plt.show()
print('Saved: figures/02_boxplots.png')

Saved: figures/02_boxplots.png


**Observation:** Box plots of the cleaned, log-transformed features show that extreme anomalies have been eliminated. Medians and interquartile ranges are now well-proportioned across all 10 features.

**Purpose:** Correlation heatmap of log-transformed features to understand feature relationships that may affect clustering.

In [8]:
fig, ax = plt.subplots(figsize=(12, 9))
corr = df_clean.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap (Log-Transformed Features)', fontsize=13)
plt.tight_layout()
plt.savefig('figures/03_correlation_heatmap.png', dpi=150)
plt.show()
print('Saved: figures/03_correlation_heatmap.png')

Saved: figures/03_correlation_heatmap.png


**Observation:** The strongest positive correlation is between Bonus_miles and cc1_miles (r = 0.84), followed by Bonus_miles and Bonus_trans (r = 0.61), reflecting credit card loyalty accrual. Flight_miles_12mo and Flight_trans_12 are also strongly correlated (r = 0.79).

**Purpose:** Pair plot of 4 to 5 key features to look for natural groupings in 2D feature projections.

In [9]:
key_candidates = ['Balance', 'Qual_miles', 'Flight_trans_12', 'Flight_miles_12mo', 'Days_since_enroll']
key_cols_pp = [c for c in key_candidates if c in df_clean.columns]
fig = sns.pairplot(df_clean[key_cols_pp], diag_kind='kde', plot_kws={'alpha': 0.4, 's': 10})
fig.figure.suptitle('Pair Plot of Key Features (log-transformed)', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig('figures/04_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/04_pairplot.png')

Saved: figures/04_pairplot.png


**Observation:** The pair plot reveals that most customers cluster near lower flight miles and transactions, while a smaller group forms a distinct high-activity travel segment with high Balance and frequent flights.

**Purpose:** PCA 2D scatter of the scaled data to visualise whether any visible groupings exist in the first two principal components, and to see how much variance they capture.

In [10]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_ * 100
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.3, s=8, color='steelblue')
ax.set_title(f'PCA 2D Scatter of Scaled Data\n(PC1={explained[0]:.1f}%, PC2={explained[1]:.1f}%)', fontsize=13)
ax.set_xlabel(f'PC1 ({explained[0]:.1f}% variance)')
ax.set_ylabel(f'PC2 ({explained[1]:.1f}% variance)')
plt.tight_layout()
plt.savefig('figures/05_pca_initial.png', dpi=150)
plt.show()
print(f'PC1={explained[0]:.1f}%, PC2={explained[1]:.1f}%, Total={sum(explained[:2]):.1f}%')

PC1=39.7%, PC2=20.8%, Total=60.6%


**Observation:** The first two principal components explain 39.73% and 20.83% of the variance respectively, totaling 60.56%. The 2D PCA projection displays a continuous cloud with a dense central core and extending arms, suggesting natural subgroupings.

## Section 3: K-Means Clustering

**Purpose:** Plot the elbow curve (inertia for K=1 to 10) and silhouette scores for K=2 to 10 to identify the optimal K. The elbow is the point where adding more clusters gives diminishing inertia reduction. The silhouette score measures how well each point is in its own cluster vs the nearest other cluster (higher is better).

In [11]:
inertias = []
silhouette_scores_km = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    if k >= 2:
        silhouette_scores_km.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(list(K_range), inertias, marker='o', color='steelblue', linewidth=2)
ax1.set_title('K-Means Elbow Curve (Inertia)')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.set_xticks(list(K_range))
ax2.plot(list(range(2, 11)), silhouette_scores_km, marker='s', color='darkorange', linewidth=2)
ax2.set_title('K-Means Silhouette Scores')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(list(range(2, 11)))
plt.suptitle('K-Means: Choosing Optimal K', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/06_kmeans_elbow_silhouette.png', dpi=150)
plt.show()
print('Inertias:', [f'{v:.0f}' for v in inertias])
print('Silhouette (K=2..10):', [f'{v:.4f}' for v in silhouette_scores_km])
best_k = list(range(2, 11))[int(np.argmax(silhouette_scores_km))]
print(f'Best K by silhouette: {best_k}')

Inertias: ['28984', '21285', '16192', '12568', '10310', '8765', '7901', '7300', '6842', '6419']
Silhouette (K=2..10): ['0.2681', '0.3174', '0.3199', '0.3001', '0.3170', '0.2984', '0.2812', '0.2761', '0.2629']
Best K by silhouette: 4


**Observation:** Inertia drops rapidly from K=1 (28,984) to K=4 (12,568), after which the rate of decrease levels off, showing an elbow around K=3 to 4. Silhouette scores peak at K=4 with a score of 0.3199 (compared to 0.3174 for K=3 and 0.3001 for K=5). Both metrics converge on K=4 as the optimal cluster count.

**Purpose:** Fit the final K-Means model with the chosen K and compute all evaluation metrics.

In [12]:
CHOSEN_K = best_k
print(f'Using K = {CHOSEN_K} (justified by elbow + silhouette evidence above)')
kmeans = KMeans(n_clusters=CHOSEN_K, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
kmeans_sil = silhouette_score(X_scaled, kmeans_labels)
kmeans_db = davies_bouldin_score(X_scaled, kmeans_labels)
kmeans_ch = calinski_harabasz_score(X_scaled, kmeans_labels)
print(f'Silhouette: {kmeans_sil:.4f}')
print(f'Davies-Bouldin: {kmeans_db:.4f}')
print(f'Calinski-Harabasz: {kmeans_ch:.2f}')
print(f'Cluster sizes: {pd.Series(kmeans_labels).value_counts().sort_index().to_dict()}')

Using K = 4 (justified by elbow + silhouette evidence above)


Silhouette: 0.3199
Davies-Bouldin: 0.9021
Calinski-Harabasz: 1575.63
Cluster sizes: {0: 1005, 1: 1630, 2: 987, 3: 1}


**Observation:** Final K-Means with K=4 yields a silhouette score of 0.3199, Davies-Bouldin index of 0.9021, and Calinski-Harabasz score of 1575.63. Cluster sizes are 1,005 (Cluster 0), 1,630 (Cluster 1), 987 (Cluster 2), and 1 (Cluster 3).

## Section 4: Hierarchical Clustering

**Purpose:** Plot truncated dendrograms for ward, complete, average, and single linkage. The truncated form shows the last 30 merges, making it readable for large datasets. Ward minimises total within-cluster variance; complete uses maximum pairwise distance; average uses average pairwise distance; single uses minimum distance (prone to chaining).

In [13]:
linkage_methods = ['ward', 'complete', 'average', 'single']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Dendrograms for Different Linkage Methods (Truncated, last 30 merges)', fontsize=13, fontweight='bold')
for ax, method in zip(axes.flatten(), linkage_methods):
    Z = linkage(X_scaled, method=method)
    dendrogram(Z, ax=ax, truncate_mode='lastp', p=30, leaf_rotation=90,
               color_threshold=0.7*max(Z[:, 2]))
    ax.set_title(f'Linkage: {method}')
    ax.set_xlabel('Sample / Cluster')
    ax.set_ylabel('Distance')
plt.tight_layout()
plt.savefig('figures/07_dendrograms.png', dpi=150)
plt.show()
print('Saved: figures/07_dendrograms.png')

Saved: figures/07_dendrograms.png


**Observation:** The truncated dendrograms show that Ward linkage produces balanced, compact cluster trees with clear distance separation. Single linkage exhibits severe chaining, while average and complete linkages generate unbalanced groupings.

**Purpose:** Compare silhouette scores for all four linkage methods at the chosen K to pick the best one. Note whether single linkage shows chaining behaviour (elongated chains rather than compact clusters).

In [14]:
hier_results = {}
for method in linkage_methods:
    agg = AgglomerativeClustering(n_clusters=CHOSEN_K, linkage=method)
    labels = agg.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    hier_results[method] = {'labels': labels, 'silhouette': sil}
hier_sil_df = pd.DataFrame([{'Linkage': m, 'Silhouette': hier_results[m]['silhouette']} for m in linkage_methods])
hier_sil_df = hier_sil_df.sort_values('Silhouette', ascending=False).reset_index(drop=True)
print(f'=== Silhouette Scores at K={CHOSEN_K} ===')
print(hier_sil_df.to_string(index=False))
best_linkage = hier_sil_df.iloc[0]['Linkage']
print(f'Best linkage: {best_linkage}')
hier_labels = hier_results[best_linkage]['labels']
hier_sil = hier_results[best_linkage]['silhouette']
hier_db = davies_bouldin_score(X_scaled, hier_labels)
hier_ch = calinski_harabasz_score(X_scaled, hier_labels)
print(f'Hierarchical ({best_linkage}): Silhouette={hier_sil:.4f}, DB={hier_db:.4f}, CH={hier_ch:.2f}')

=== Silhouette Scores at K=4 ===
 Linkage  Silhouette
    ward    0.331692
  single    0.264134
 average    0.251780
complete    0.245289
Best linkage: ward
Hierarchical (ward): Silhouette=0.3317, DB=0.8735, CH=1410.12


**Observation:** At K=4, Ward linkage achieves the highest silhouette score (0.3317), followed by single (0.2641), average (0.2518), and complete (0.2453). Ward linkage is selected as the best hierarchical model, with Davies-Bouldin of 0.8735 and Calinski-Harabasz of 1410.12.

## Section 5: DBSCAN Clustering

**Purpose:** Plot the k-distance graph to identify the elbow point for eps. For each point we find its k-th nearest neighbour distance. Sorted in descending order, the elbow of this curve is a good estimate for eps, the neighbourhood radius in DBSCAN.

In [15]:
k_nn = 5
nbrs = NearestNeighbors(n_neighbors=k_nn).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])[::-1]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_distances, color='steelblue', linewidth=1.5)
ax.set_title(f'{k_nn}-Distance Graph for DBSCAN eps Selection')
ax.set_xlabel('Points sorted by distance (descending)')
ax.set_ylabel(f'{k_nn}th Nearest Neighbour Distance')
plt.tight_layout()
plt.savefig('figures/08_kdistance_plot.png', dpi=150)
plt.show()
knee_idx = int(0.05 * len(k_distances))
eps_knee = k_distances[knee_idx]
print(f'Approximate knee eps value: {eps_knee:.4f}')

Approximate knee eps value: 0.9413


**Observation:** The sorted 5-nearest neighbour distance curve shows a sharp knee around distance 0.94, providing the baseline search region for the DBSCAN neighbourhood radius parameter eps.

**Purpose:** Grid search over eps values around the knee and min_samples values (5, 10, 2*n_features). Silhouette for DBSCAN is computed on non-noise points only (noise points with label -1 are excluded because silhouette requires cluster assignments).

In [16]:
n_features = X_scaled.shape[1]
eps_values = np.round(np.linspace(eps_knee * 0.5, eps_knee * 1.8, 7), 3)
min_samples_values = [5, 10, 2 * n_features]
dbscan_results = []
for eps in eps_values:
    for min_s in min_samples_values:
        db = DBSCAN(eps=float(eps), min_samples=int(min_s))
        labels = db.fit_predict(X_scaled)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = int((labels == -1).sum())
        noise_pct = round(n_noise / len(labels) * 100, 2)
        non_noise = labels != -1
        if n_clusters >= 2 and non_noise.sum() > 1:
            sil = round(float(silhouette_score(X_scaled[non_noise], labels[non_noise])), 4)
        else:
            sil = None
        dbscan_results.append({'eps': round(float(eps), 3), 'min_samples': int(min_s),
                                'n_clusters': n_clusters, 'n_noise': n_noise,
                                'noise_pct': noise_pct, 'silhouette': sil})
dbscan_df = pd.DataFrame(dbscan_results)
print('=== DBSCAN Grid Search ===')
print(dbscan_df.to_string(index=False))

=== DBSCAN Grid Search ===
  eps  min_samples  n_clusters  n_noise  noise_pct  silhouette
0.471            5          28     1159      31.99      0.0547
0.471           10          12     1575      43.47      0.0302
0.471           20           7     2162      59.67      0.1573
0.675            5          20      432      11.92      0.1118
0.675           10          16      793      21.89      0.1609
0.675           20           7     1263      34.86      0.1760
0.879            5           4      112       3.09      0.2889
0.879           10           3      220       6.07      0.3353
0.879           20           4      455      12.56      0.3123
1.082            5           2       19       0.52      0.3058
1.082           10           3       33       0.91      0.3335
1.082           20           3      105       2.90      0.3345
1.286            5           2        7       0.19      0.3057
1.286           10           2        9       0.25      0.3057
1.286           20          

**Observation:** Grid search across eps (0.471 to 1.694) and min_samples (5, 10, 20) reveals that low eps values leave over 30% of points as noise, while high eps merges points into a single cluster. The parameter set eps=0.879, min_samples=10 yields 3 dense clusters with 220 noise points (6.07%) and non-noise silhouette score of 0.3353.

**Purpose:** Select best DBSCAN parameters (at least 2 clusters, noise <= 30%, highest silhouette) and fit final model.

In [17]:
valid = dbscan_df[(dbscan_df['n_clusters'] >= 2) & (dbscan_df['noise_pct'] <= 30) & (dbscan_df['silhouette'].notna())]
if len(valid) == 0:
    valid = dbscan_df[(dbscan_df['n_clusters'] >= 2) & (dbscan_df['silhouette'].notna())]
best_row = valid.loc[valid['silhouette'].idxmax()]
BEST_EPS = float(best_row['eps'])
BEST_MIN_SAMPLES = int(best_row['min_samples'])
print(f'Best DBSCAN: eps={BEST_EPS}, min_samples={BEST_MIN_SAMPLES}')

dbscan_final = DBSCAN(eps=BEST_EPS, min_samples=BEST_MIN_SAMPLES)
dbscan_labels = dbscan_final.fit_predict(X_scaled)
n_db_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_db_noise = int((dbscan_labels == -1).sum())
non_noise = dbscan_labels != -1
if n_db_clusters >= 2 and non_noise.sum() > 1:
    dbscan_sil = float(silhouette_score(X_scaled[non_noise], dbscan_labels[non_noise]))
    dbscan_db = float(davies_bouldin_score(X_scaled[non_noise], dbscan_labels[non_noise]))
    dbscan_ch = float(calinski_harabasz_score(X_scaled[non_noise], dbscan_labels[non_noise]))
else:
    dbscan_sil = dbscan_db = dbscan_ch = float('nan')
print(f'n_clusters={n_db_clusters}, n_noise={n_db_noise} ({n_db_noise/len(dbscan_labels)*100:.1f}%)')
print(f'Silhouette={dbscan_sil:.4f}, DB={dbscan_db:.4f}, CH={dbscan_ch:.2f}')

Best DBSCAN: eps=0.879, min_samples=10


n_clusters=3, n_noise=220 (6.1%)
Silhouette=0.3353, DB=1.1554, CH=1512.32


**Observation:** Fitting DBSCAN with eps=0.879 and min_samples=10 identifies 3 dense customer clusters and marks 220 records (6.07%) as noise. Non-noise silhouette score is 0.3353, Davies-Bouldin is 1.1554, and Calinski-Harabasz is 1512.32.

## Section 6: Visualization

**Purpose:** PCA 2D scatter for each final model. DBSCAN noise points are shown in grey.

In [18]:
colors = plt.cm.tab10.colors
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('PCA 2D Cluster Visualizations', fontsize=14, fontweight='bold')
for cid in np.unique(kmeans_labels):
    m = kmeans_labels == cid
    axes[0].scatter(X_pca[m, 0], X_pca[m, 1], s=8, alpha=0.5, color=colors[cid % 10], label=f'C{cid}')
axes[0].set_title(f'K-Means (K={CHOSEN_K})')
axes[0].set_xlabel(f'PC1 ({explained[0]:.1f}%)')
axes[0].set_ylabel(f'PC2 ({explained[1]:.1f}%)')
axes[0].legend(markerscale=3, fontsize=8)
for cid in np.unique(hier_labels):
    m = hier_labels == cid
    axes[1].scatter(X_pca[m, 0], X_pca[m, 1], s=8, alpha=0.5, color=colors[cid % 10], label=f'C{cid}')
axes[1].set_title(f'Hierarchical ({best_linkage}, K={CHOSEN_K})')
axes[1].set_xlabel(f'PC1 ({explained[0]:.1f}%)')
axes[1].set_ylabel(f'PC2 ({explained[1]:.1f}%)')
axes[1].legend(markerscale=3, fontsize=8)
for cid in np.unique(dbscan_labels):
    m = dbscan_labels == cid
    c = 'grey' if cid == -1 else colors[cid % 10]
    lbl = 'Noise' if cid == -1 else f'C{cid}'
    axes[2].scatter(X_pca[m, 0], X_pca[m, 1], s=8, alpha=0.4 if cid == -1 else 0.5, color=c, label=lbl)
axes[2].set_title(f'DBSCAN (eps={BEST_EPS}, min_s={BEST_MIN_SAMPLES})')
axes[2].set_xlabel(f'PC1 ({explained[0]:.1f}%)')
axes[2].set_ylabel(f'PC2 ({explained[1]:.1f}%)')
axes[2].legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.savefig('figures/09_cluster_pca_scatter.png', dpi=150)
plt.show()
print('Saved: figures/09_cluster_pca_scatter.png')

Saved: figures/09_cluster_pca_scatter.png


**Observation:** The 2D PCA plots illustrate the distinct geometric behavior of each algorithm. K-Means and Hierarchical partition the entire space into compact regions, while DBSCAN identifies the dense core groups and isolates peripheral outliers in grey.

## Section 7: Evaluation

**Purpose:** Comparison table of all three models. Silhouette score (range -1 to 1, higher is better) measures separation. Davies-Bouldin (lower is better) penalises compact but close clusters. Calinski-Harabasz (higher is better) is the ratio of between-cluster to within-cluster dispersion.

In [19]:
comparison = pd.DataFrame([
    {'Algorithm': 'K-Means', 'N_Clusters': int(CHOSEN_K), 'Noise_Points': 0, 'Noise_Pct': 0.0,
     'Silhouette': round(kmeans_sil, 4), 'Davies_Bouldin': round(kmeans_db, 4), 'Calinski_Harabasz': round(kmeans_ch, 2)},
    {'Algorithm': f'Hierarchical ({best_linkage})', 'N_Clusters': int(CHOSEN_K), 'Noise_Points': 0, 'Noise_Pct': 0.0,
     'Silhouette': round(hier_sil, 4), 'Davies_Bouldin': round(hier_db, 4), 'Calinski_Harabasz': round(hier_ch, 2)},
    {'Algorithm': 'DBSCAN', 'N_Clusters': int(n_db_clusters), 'Noise_Points': int(n_db_noise),
     'Noise_Pct': round(n_db_noise/len(dbscan_labels)*100, 2),
     'Silhouette': round(dbscan_sil, 4) if not (dbscan_sil != dbscan_sil) else None,
     'Davies_Bouldin': round(dbscan_db, 4) if not (dbscan_db != dbscan_db) else None,
     'Calinski_Harabasz': round(dbscan_ch, 2) if not (dbscan_ch != dbscan_ch) else None}
])
print('=== Clustering Comparison Table ===')
print(comparison.to_string(index=False))
comparison.to_csv('results/clustering_comparison.csv', index=False)
print('Saved: results/clustering_comparison.csv')

=== Clustering Comparison Table ===
          Algorithm  N_Clusters  Noise_Points  Noise_Pct  Silhouette  Davies_Bouldin  Calinski_Harabasz
            K-Means           4             0       0.00      0.3199          0.9021            1575.63
Hierarchical (ward)           4             0       0.00      0.3317          0.8735            1410.12
             DBSCAN           3           220       6.07      0.3353          1.1554            1512.32
Saved: results/clustering_comparison.csv


**Observation:** Hierarchical (Ward) achieved the highest silhouette (0.3317) and lowest Davies-Bouldin (0.8735) among full-partition models. K-Means achieved 0.3199 silhouette with highest Calinski-Harabasz (1575.63). DBSCAN achieved 0.3353 on non-noise points while setting aside 6.07% noise.

## Section 8: Cluster Analysis and Interpretation

**Purpose:** Profile K-Means clusters using mean ORIGINAL (unscaled) feature values and Award? rate. We map back to original values using df_features filtered to the same rows that survived outlier removal.

In [20]:
z_scores_orig = np.abs(stats.zscore(df_transformed))
z_mask_orig = (z_scores_orig < 3).all(axis=1)
df_orig_clean = df_features[z_mask_orig].reset_index(drop=True)

profile_df = df_orig_clean.copy()
profile_df['KMeans_Cluster'] = kmeans_labels
profile_df['Award'] = award_clean.values

print('=== Cluster Sizes ===')
print(profile_df['KMeans_Cluster'].value_counts().sort_index())
print()
cluster_profile = profile_df.groupby('KMeans_Cluster').mean()
print('=== Mean Cluster Profile (original scale) ===')
print(cluster_profile.to_string())
cluster_profile.to_csv('results/kmeans_cluster_profile.csv')
print('Saved: results/kmeans_cluster_profile.csv')

=== Cluster Sizes ===
KMeans_Cluster
0    1005
1    1630
2     987
3       1
Name: count, dtype: int64

=== Mean Cluster Profile (original scale) ===
                      Balance  Qual_miles  cc1_miles  cc2_miles  cc3_miles   Bonus_miles  Bonus_trans  Flight_miles_12mo  Flight_trans_12  Days_since_enroll     Award
KMeans_Cluster                                                                                                                                                        
0               103835.889552         0.0   2.352239        1.0        1.0  24034.951244    14.534328        1075.540299         3.029851        4423.163184  0.542289
1                75965.036810         0.0   2.525153        1.0        1.0  20900.167485    14.396319           0.000000         0.000000        4349.069939  0.361963
2                22418.097264         0.0   1.000000        1.0        1.0    789.028369     1.787234          10.539007         0.038501        3392.637285  0.117528
3              

**Observation:** Cluster 0 (1,005 members) averages 103,836 miles, 1,076 flight miles in 12 months, and 54.2% award rate. Cluster 1 (1,630 members) averages 75,965 miles, 20,900 bonus miles via credit cards, but zero flights in the past year (36.2% award rate). Cluster 2 (987 members) averages 22,418 miles, only 789 bonus miles, and 11.8% award rate.

**Purpose:** Heatmap of standardized cluster means to visually see which features distinguish each cluster.

In [21]:
profile_std = (cluster_profile - cluster_profile.mean()) / cluster_profile.std()
fig, ax = plt.subplots(figsize=(12, max(5, CHOSEN_K + 2)))
sns.heatmap(profile_std.T, annot=True, fmt='.2f', cmap='RdYlGn', center=0, linewidths=0.5, ax=ax)
ax.set_title('Standardized Cluster Means (K-Means)', fontsize=13)
ax.set_xlabel('Cluster')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig('figures/10_cluster_heatmap.png', dpi=150)
plt.show()
print('Saved: figures/10_cluster_heatmap.png')

Saved: figures/10_cluster_heatmap.png


**Observation:** The standardized cluster heatmap visually contrasts the behavioral profiles: Cluster 0 scores highest on flight activity, Cluster 1 scores highest on credit card mileage, and Cluster 2 is below average across all engagement metrics.

**Purpose:** Assign business names to each cluster based on the profile. Display the key features that drive each name.

In [22]:
key_cols_profile = [c for c in ['Balance', 'Qual_miles', 'Flight_miles_12mo', 'Flight_trans_12',
                                    'Days_since_enroll', 'Award'] if c in cluster_profile.columns]
print('=== Key Features for Business Naming ===')
print(cluster_profile[key_cols_profile].round(2).to_string())
print()
print('Business segment names are assigned from the profile above after execution.')
print('Guidelines used:')
print('  High Balance + High Qual_miles + High Award -> Elite / Frequent Flyers')
print('  Low flights + Low Balance + Long enroll     -> Dormant / Lapsed Members')
print('  Medium metrics                              -> Occasional Travellers')
print('  High recent flights + Low Balance           -> Active New Members')

=== Key Features for Business Naming ===
                  Balance  Qual_miles  Flight_miles_12mo  Flight_trans_12  Days_since_enroll  Award
KMeans_Cluster                                                                                     
0               103835.89         0.0            1075.54             3.03            4423.16   0.54
1                75965.04         0.0               0.00             0.00            4349.07   0.36
2                22418.10         0.0              10.54             0.04            3392.64   0.12
3                61405.00       250.0            1350.00             8.00            4828.00   1.00

Business segment names are assigned from the profile above after execution.
Guidelines used:
  High Balance + High Qual_miles + High Award -> Elite / Frequent Flyers
  Low flights + Low Balance + Long enroll     -> Dormant / Lapsed Members
  Medium metrics                              -> Occasional Travellers
  High recent flights + Low Balance           -

**Observation:** Cluster 0: 'Active Frequent Flyers' (high flights, high balance, 54% award rate). Cluster 1: 'Credit Card Loyalty Earners' (high cc/bonus miles, zero recent flights). Cluster 2: 'Infrequent / Inactive Members' (low balances, low activity, 12% award rate). Cluster 3: 'Outlier Account' (single account with exceptional qualifying miles).

**Purpose:** Compare how the three algorithms segment customers and give marketing recommendations.

In [23]:
print('=== Algorithm Comparison ===')
print(f'K-Means (K={CHOSEN_K}): All points assigned. Good for compact spherical clusters.')
print(f'Hierarchical ({best_linkage}): Tree-based. Captures hierarchical relationships.')
print(f'DBSCAN: Found {n_db_clusters} dense clusters, {n_db_noise} noise ({n_db_noise/len(dbscan_labels)*100:.1f}%) points.')
print()
print('=== Marketing Recommendations ===')
print('1. Frequent/Elite flyers (high Balance, Qual_miles, Award): Premium upgrades, VIP lounges, bonus miles.')
print('2. Dormant members (low flights, long since enroll): Re-engagement promotions, bonus miles on next flight.')
print('3. Occasional travellers (mid metrics): Targeted deals to increase frequency.')
print('4. DBSCAN noise points: Possibly unique high-value outliers needing personalised outreach.')

=== Algorithm Comparison ===
K-Means (K=4): All points assigned. Good for compact spherical clusters.
Hierarchical (ward): Tree-based. Captures hierarchical relationships.
DBSCAN: Found 3 dense clusters, 220 noise (6.1%) points.

=== Marketing Recommendations ===
1. Frequent/Elite flyers (high Balance, Qual_miles, Award): Premium upgrades, VIP lounges, bonus miles.
2. Dormant members (low flights, long since enroll): Re-engagement promotions, bonus miles on next flight.
3. Occasional travellers (mid metrics): Targeted deals to increase frequency.
4. DBSCAN noise points: Possibly unique high-value outliers needing personalised outreach.


**Observation:** Marketing recommendations: Offer tiered flight upgrade bonuses to Active Frequent Flyers; incentivize Credit Card Earners to redeem miles on actual flights; launch re-engagement bonus campaigns for Inactive Members to prevent churn.

**Purpose:** Save all key metrics to results/metrics.json for use in the PDF report.

In [24]:
metrics = {
    'chosen_k': int(CHOSEN_K),
    'best_linkage': best_linkage,
    'best_dbscan_eps': float(BEST_EPS),
    'best_dbscan_min_samples': int(BEST_MIN_SAMPLES),
    'n_rows_after_outlier_removal': int(len(df_clean)),
    'n_rows_removed': int(n_removed),
    'pct_removed': float(round(pct_removed, 2)),
    'skewed_cols': skewed_cols,
    'pca_explained_variance': [round(float(e), 2) for e in explained[:2]],
    'kmeans': {'silhouette': round(kmeans_sil, 4), 'davies_bouldin': round(kmeans_db, 4), 'calinski_harabasz': round(kmeans_ch, 2)},
    'hierarchical': {'silhouette': round(hier_sil, 4), 'davies_bouldin': round(hier_db, 4), 'calinski_harabasz': round(hier_ch, 2)},
    'dbscan': {'n_clusters': int(n_db_clusters), 'n_noise': int(n_db_noise),
               'noise_pct': round(n_db_noise/len(dbscan_labels)*100, 2),
               'silhouette': None if (dbscan_sil != dbscan_sil) else round(dbscan_sil, 4),
               'davies_bouldin': None if (dbscan_db != dbscan_db) else round(dbscan_db, 4),
               'calinski_harabasz': None if (dbscan_ch != dbscan_ch) else round(dbscan_ch, 2)}
}
with open('results/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
skew_table.to_csv('results/skewness_table.csv', index=False)
hier_sil_df.to_csv('results/hierarchical_linkage_comparison.csv', index=False)
dbscan_df.to_csv('results/dbscan_grid.csv', index=False)
print('All results saved to results/')
print(json.dumps(metrics, indent=2))

All results saved to results/
{
  "chosen_k": 4,
  "best_linkage": "ward",
  "best_dbscan_eps": 0.879,
  "best_dbscan_min_samples": 10,
  "n_rows_after_outlier_removal": 3623,
  "n_rows_removed": 376,
  "pct_removed": 9.4,
  "skewed_cols": [
    "cc3_miles",
    "cc2_miles",
    "Qual_miles",
    "Flight_miles_12mo",
    "Flight_trans_12",
    "Balance",
    "Bonus_miles",
    "Bonus_trans"
  ],
  "pca_explained_variance": [
    39.73,
    20.83
  ],
  "kmeans": {
    "silhouette": 0.3199,
    "davies_bouldin": 0.9021,
    "calinski_harabasz": 1575.63
  },
  "hierarchical": {
    "silhouette": 0.3317,
    "davies_bouldin": 0.8735,
    "calinski_harabasz": 1410.12
  },
  "dbscan": {
    "n_clusters": 3,
    "n_noise": 220,
    "noise_pct": 6.07,
    "silhouette": 0.3353,
    "davies_bouldin": 1.1554,
    "calinski_harabasz": 1512.32
  }
}


**Observation:** All figures and results tables have been exported to figures/ and results/ for inclusion in the formal PDF report.

## Section 9: Conclusion and Limitations

**Conclusion:**

This study segmented East-West Airlines frequent flyer customers using three clustering algorithms. K-Means produced clear, interpretable segments after log-transformation and StandardScaler preprocessing. The optimal K was chosen from the elbow curve (inertia) and silhouette score together. Hierarchical clustering confirmed similar groupings; the dendrogram showed how clusters relate at different distance thresholds. DBSCAN identified dense core segments while marking a fraction of customers as noise, indicating some customers have unique travel patterns not fitting any dense group.

**Limitations:**

1. The choice of K is partly subjective; elbow and silhouette may suggest different values on a different dataset.
2. Results depend on the log-transformation and StandardScaler; different preprocessing would shift cluster boundaries.
3. DBSCAN struggles with varying density in this data; the high-skew features make uniform density assumptions difficult.
4. PCA 2D visualisation loses information from remaining components; true high-dimensional structure may differ from 2D projections.
5. cc1/cc2/cc3_miles are kept as ordinal numeric, but they represent categorical tiers; different encoding could alter results.

## Attribution

Code structure drafted with AI assistance (Claude); executed, verified and interpreted by <NAME>. Dataset source: Shmueli, G. and Bruce, P. (2016). Data Mining for Business Analytics. East-West Airlines dataset (company names changed).